# Two-minus sector: closed-form $A_n$ for 1-D deep-water waves

**Result.** For the two-minus sector $\sigma=(-1,-1,+1,\dots,+1)$ (legs $1,2$ are
the "minus" legs), the tree amplitude is

$$\boxed{\,A_n \;=\; i\,\cdot 2^{\,n-1}\,g^{\,3-n}\,\big(\omega_1\omega_2\big)\,
\big[\min(\omega_1^2,\omega_2^2)\big]^{\,n-3}\,}$$

equivalently, with $\omega_<,\omega_>$ the smaller/larger-$|\cdot|$ minus-leg
frequencies, $\;A_n = i\,2^{n-1}g^{3-n}\,\omega_>\,\omega_<^{\,2n-5}$.

It depends only on the two minus legs (the plus legs enter solely through the
on-shell constraints).  This cell loads a self-contained Python port of the
Berends–Giele recursion in `OnShellBG.m` and the closed form.

In [1]:
import mpmath as mp
from fractions import Fraction as F
from waterwave_bg import (bg_amplitude_hp, closed_form_A, two_minus_kinematics,
                          in_physical_regime)
mp.mp.dps = 50

def report(n, free_w, g=1, label=""):
    k, w, sig = two_minus_kinematics(n, [F(x) for x in free_w], F(g))
    A_bg  = bg_amplitude_hp(k, w, F(g), dps=50)
    A_cf  = closed_form_A(w, sig, F(g))
    rel   = abs(A_bg - A_cf)/abs(A_bg)
    ok    = rel < mp.mpf(10)**-10
    print(f"  {label:22s} n={n} g={str(g):4s}  BG={mp.nstr(A_bg.imag,10)}i  "
          f"closed={mp.nstr(mp.mpf(A_cf.imag),10)}i  rel={mp.nstr(rel,2)}  "
          f"{'PASS' if ok else 'FAIL'}")
    return ok


## 1. Standard kinematics (the patterns used in `OnShellBG.m`'s tests)

In [2]:
ok = []
ok.append(report(5, [3/2, 2, 5/2],          label="A5 {3/2,2,5/2}"))
ok.append(report(5, [1, 3, 5],              label="A5 {1,3,5}"))
ok.append(report(6, [3/2, 2, 5/2, 3],       label="A6 {3/2,2,5/2,3}"))
ok.append(report(6, [1, 3, 5, 7],           label="A6 {1,3,5,7}"))
ok.append(report(7, [3/2, 2, 5/2, 3, 7/2],  label="A7 {3/2,2,5/2,3,7/2}"))
ok.append(report(7, [1, 2, 3, 5, 7],        label="A7 {1,2,3,5,7}"))
print("all pass:", all(ok))


  A5 {3/2,2,5/2}         n=5 g=1     BG=-445.5i  closed=-445.5i  rel=2.0e-50  PASS
  A5 {1,3,5}             n=5 g=1     BG=-101.3333333i  closed=-101.3333333i  rel=4.7e-17  PASS
  A6 {3/2,2,5/2,3}       n=6 g=1     BG=-2976.75i  closed=-2976.75i  rel=1.7e-48  PASS
  A6 {1,3,5,7}           n=6 g=1     BG=-338.0i  closed=-338.0i  rel=2.2e-45  PASS
  A7 {3/2,2,5/2,3,7/2}   n=7 g=1     BG=-18255.9825i  closed=-18255.9825i  rel=1.0e-16  PASS
  A7 {1,2,3,5,7}         n=7 g=1     BG=-728.8888889i  closed=-728.8888889i  rel=3.5e-17  PASS
all pass: True


## 2. Non-generic regimes
"one frequency much larger / much smaller than the others" — kept in the
physical regime (a minus leg carries the smallest momentum).

In [3]:
ok = []
ok.append(report(5, [2, 3, 1000],   label="plus leg huge"))      # plus 1000 >> rest
ok.append(report(6, [3/2,2,5/2,5000],label="plus leg huge"))
ok.append(report(7, [1,2,5/2,3,10**4],label="plus leg huge"))
# a minus leg made tiny: feed it as the first free frequency (w2 = minus leg)
ok.append(report(5, [F(1,1000), 3, 5], label="minus leg tiny"))
ok.append(report(6, [F(1,500), 3, 5, 7], label="minus leg tiny"))
print("all pass:", all(ok))


  plus leg huge          n=5 g=1     BG=-512007.6418i  closed=-512007.6418i  rel=3.2e-17  PASS
  plus leg huge          n=6 g=1     BG=-2733752.403i  closed=-2733752.403i  rel=1.2e-16  PASS
  plus leg huge          n=7 g=1     BG=-640000.2894i  closed=-640000.2894i  rel=1.0e-16  PASS
  minus leg tiny         n=5 g=1     BG=-9.800374953e-14i  closed=-9.800374953e-14i  rel=1.2e-17  PASS
  minus leg tiny         n=6 g=1     BG=-4.205485135e-17i  closed=-4.205485135e-17i  rel=4.3e-17  PASS
all pass: True


## 3. Coupling dependence $g\neq 1$  (checks the $g^{3-n}$ factor)

In [4]:
ok = []
ok.append(report(5, [1,3,5], g=2,        label="g=2"))
ok.append(report(5, [2,3,5], g=F(7,3),   label="g=7/3"))
ok.append(report(6, [1,3,5,7], g=2,      label="g=2"))
ok.append(report(7, [1,2,3,5,7], g=5,    label="g=5"))
print("all pass:", all(ok))


  g=2                    n=5 g=2     BG=-25.33333333i  closed=-25.33333333i  rel=4.7e-17  PASS
  g=7/3                  n=5 g=7/3   BG=-611.2653061i  closed=-611.2653061i  rel=8.7e-17  PASS
  g=2                    n=6 g=2     BG=-42.25i  closed=-42.25i  rel=2.2e-45  PASS
  g=5                    n=7 g=5     BG=-1.166222222i  closed=-1.166222222i  rel=4.2e-17  PASS
all pass: True


## 4. The $n=4$ base case (limit)
At $n=4$ the two-minus on-shell manifold collapses to $\{\omega_1,\omega_2\}=
\{-\omega_3,-\omega_4\}$, which puts an internal line exactly on-shell ($0/0$
in the propagator).  Approaching it off the momentum-conservation surface
($\epsilon\to0$) gives a finite limit equal to the closed form
$A_4=i\,2^{3}g^{-1}\,\omega_1\omega_2\min(\omega_1^2,\omega_2^2)$.

In [5]:
from waterwave_bg import bg_amplitude
def A4_limit(w3, w4, eps):
    # deform off momentum-conservation: w = (-w3, -w4-eps, w3+eps, w4)
    w = [-w3, -w4-eps, w3+eps, w4]
    k = [s*x*x for s, x in zip([-1,-1,1,1], w)]
    return bg_amplitude(k, w, 1)
for (w3, w4) in [(3,2),(5,2),(7,3)]:
    cf = 1j*2**3*( (-w3)*(-w4) )*min(w3**2, w4**2)   # closed form, g=1
    print(f"  w3={w3} w4={w4}: closed={cf.imag:.0f}i   limit:",
          [f"{A4_limit(w3,w4,e).imag:.4f}" for e in (1e-2,1e-3,1e-4,1e-5)])


  w3=3 w4=2: closed=192i   limit: ['192.8204', '192.0820', '192.0082', '192.0008']
  w3=5 w4=2: closed=320i   limit: ['321.6636', '320.1660', '320.0166', '320.0017']
  w3=7 w4=3: closed=1512i   limit: ['1516.8864', '1512.4881', '1512.0488', '1512.0049']


## 5. Domain note
The closed form is exact whenever a **minus** leg carries the smallest momentum
(`in_physical_regime`).  This covers all the kinematics above.  Because the
Berends–Giele kernels contain $|k|$ (the dispersion is $\omega^2=g|k|$), the
full amplitude is *piecewise*-rational: in the (non-physical) chambers where a
**plus** leg is the softest, $A_n$ takes a different rational form.  The cell
below flags the regime.

In [6]:
for fw in [[2,3,5],[2,3,F(1,1000)]]:
    k,w,sig = two_minus_kinematics(5,[F(x) for x in fw],1)
    print(f"  free={fw}: physical regime (minus leg softest)? {in_physical_regime(w,sig)}")


  free=[2, 3, 5]: physical regime (minus leg softest)? True
  free=[2, 3, Fraction(1, 1000)]: physical regime (minus leg softest)? False
